# Step 7 (experiment): Self-training with pseudo-labels
**Idea (suggested in the organisers' Notebook 4, "Ideas for improvement"):** after learning from the n real Amsterdam labels, the model labels the unlabelled Amsterdam pixels it is **very sure** about ("pseudo-labels"), adds them to the training set with a lower weight, and retrains once. It uses unlabelled Amsterdam data, the same kind of data our own scaling already uses, and never any extra true labels.

**Rules fixed BEFORE running (so we cannot fool ourselves):**
* Settings: one round; per class at most `K_PER_CLASS = 200` pseudo-labels, only pixels with probability at least 0.5 for that class; pseudo-labels count half as much as a real label (`PSEUDO_WEIGHT = 0.5`). No tuning of these after seeing results.
* Fair test: the non-support Amsterdam pixels are split at random into an **unlabelled pool** (pseudo-labels come only from here) and an **evaluation set** (never used for anything but scoring). Baseline blend3 and self-training are scored on the SAME evaluation pixels, same support samples, same 10 seeds.
* **Adopt only if:** n=25 is better in at least 8 of 10 seeds, AND no other n gets worse on average by more than 0.005, AND it later holds up in the dress rehearsal. Otherwise it goes on the slides as "tried, did not help".

Nothing official is changed: results go to `results/self_training.csv` only. Run time: about 45 to 60 minutes.

In [1]:
import time, importlib
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import transfer_v3 as tv3
importlib.reload(tv3)

SEED, CLASSES = tv3.SEED, tv3.CLASSES
SHOTS, TRIALS = (5, 25, 50, 100, 200), 10
K_PER_CLASS, MIN_PROB, PSEUDO_WEIGHT = 200, 0.5, 0.5      # fixed in advance, do not tune

data = tv3.load_data(".")
stage1 = tv3.train_stage1(data)                            # same Stage 1 as the official run
Xa, Za, Zst, ym = data["Xa"], stage1["Za"], stage1["Zst"], data["ym"]


def blend3_fit(idx, y, rel_w, n):
    """blend3 exactly as transfer_v3.adapt_to_amsterdam, but each Amsterdam training pixel can carry its own
    relative weight (1 = real label, PSEUDO_WEIGHT = pseudo-label). With all weights 1 it IS the official blend3."""
    w = max(1.0, len(ym) / (len(CLASSES) * n) * 0.0625)   # same weight rule as tv3.finetune_proba
    X = np.vstack([data["Xm"], Xa[idx]])
    yy = np.concatenate([ym, y])
    sw = np.concatenate([np.ones(len(ym)), w * rel_w])
    rf = RandomForestClassifier(**dict(tv3.RF_PARAMS, n_estimators=150)).fit(X, yy, sample_weight=sw)
    lr = LogisticRegression(C=tv3._c_for(n), max_iter=5000).fit(Zst[idx], y, sample_weight=rel_w)
    protos = np.stack([np.average(Za[idx][y == c], axis=0, weights=rel_w[y == c]) for c in CLASSES])
    return rf, lr, protos


def blend3_proba(model, rows):
    rf, lr, protos = model
    d = np.sqrt(((Za[rows][:, None, :] - protos[None]) ** 2).sum(2))
    return (rf.predict_proba(Xa[rows]) + lr.predict_proba(Zst[rows]) + tv3._soft(d)) / 3

print("setup OK")

setup OK


In [2]:
t0 = time.time()
rows = []
for n in SHOTS:
    t = time.time()
    for k in range(TRIALS):
        seed = SEED + k
        sup, rest = tv3.sample_support(data["ya"], n, seed)          # same support as the official run
        rng = np.random.default_rng(1000 + seed)
        rest = rng.permutation(rest)
        pool, ev = rest[: len(rest) // 2], rest[len(rest) // 2:]     # unlabelled pool / evaluation set
        ys, y_ev = data["ya"][sup], data["ya"][ev]

        base = blend3_fit(sup, ys, np.ones(len(sup)), n)
        f1_base = f1_score(y_ev, CLASSES[blend3_proba(base, ev).argmax(1)], average="macro")

        if n == SHOTS[0] and k == 0:   # check: with weights 1 this must equal the official blend3 code
            _, m_off = tv3.adapt_to_amsterdam(data, stage1, n, seed, "blend3")
            P_off = tv3.predict_proba_blend3(m_off, {"rf": stage1["rf"], "scaler_a": stage1["scaler_a"]}, Xa[ev])
            f1_off = f1_score(y_ev, CLASSES[P_off.argmax(1)], average="macro")
            print(f"check vs official blend3: {f1_base:.6f} vs {f1_off:.6f}")
            assert abs(f1_base - f1_off) < 1e-6, "re-implementation differs from transfer_v3"

        # pseudo-labels: per class, the pool pixels most confidently predicted as that class
        P_pool = blend3_proba(base, pool)
        pred, conf = CLASSES[P_pool.argmax(1)], P_pool.max(1)
        pick = []
        for c in CLASSES:
            cand = np.where((pred == c) & (conf >= MIN_PROB))[0]
            pick.append(cand[np.argsort(-conf[cand])][:K_PER_CLASS])
        pick = np.concatenate(pick)
        ps_idx, ps_y = pool[pick], pred[pick]
        ps_correct = (ps_y == data["ya"][ps_idx]).mean() if len(pick) else np.nan   # for the log only

        idx = np.concatenate([sup, ps_idx]); y = np.concatenate([ys, ps_y])
        rel_w = np.concatenate([np.ones(len(sup)), np.full(len(ps_idx), PSEUDO_WEIGHT)])
        if len(set(y)) == len(CLASSES):
            st = blend3_fit(idx, y, rel_w, n)
            f1_self = f1_score(y_ev, CLASSES[blend3_proba(st, ev).argmax(1)], average="macro")
        else:
            f1_self = f1_base
        rows.append({"n_per_class": n, "seed": seed, "f1_blend3": f1_base, "f1_self_training": f1_self,
                     "n_pseudo": len(pick), "pseudo_label_accuracy": ps_correct})
    r = pd.DataFrame(rows).query("n_per_class == @n")
    print(f"n={n:3d}: blend3 {r.f1_blend3.mean():.4f} | self-training {r.f1_self_training.mean():.4f} "
          f"| better in {(r.f1_self_training > r.f1_blend3).sum()}/{TRIALS} | pseudo-labels {r.n_pseudo.mean():.0f}, "
          f"{100 * r.pseudo_label_accuracy.mean():.0f}% correct  ({time.time()-t:.0f}s)")
res = pd.DataFrame(rows)
res.round(4).to_csv("results/self_training_all_runs.csv", index=False)
print(f"done in {time.time()-t0:.0f}s")

check vs official blend3: 0.521193 vs 0.521193
n=  5: blend3 0.5801 | self-training 0.5675 | better in 1/10 | pseudo-labels 762, 82% correct  (185s)
n= 25: blend3 0.6412 | self-training 0.6372 | better in 1/10 | pseudo-labels 800, 91% correct  (203s)
n= 50: blend3 0.6653 | self-training 0.6633 | better in 3/10 | pseudo-labels 800, 93% correct  (208s)
n=200: blend3 0.7028 | self-training 0.7011 | better in 2/10 | pseudo-labels 800, 96% correct  (217s)
done in 1024s


## Verdict (rule fixed in advance)

In [3]:
s = (res.assign(diff=res.f1_self_training - res.f1_blend3, win=res.f1_self_training > res.f1_blend3)
     .groupby("n_per_class").agg(blend3=("f1_blend3", "mean"), blend3_std=("f1_blend3", "std"),
                                 self_training=("f1_self_training", "mean"), self_std=("f1_self_training", "std"),
                                 mean_diff=("diff", "mean"), wins=("win", "sum"),
                                 pseudo_acc=("pseudo_label_accuracy", "mean")).reset_index())
display(s.round(4))
s.round(4).to_csv("results/self_training.csv", index=False)

wins25 = int(s.loc[s.n_per_class == 25, "wins"].iloc[0])
others_ok = (s.loc[s.n_per_class != 25, "mean_diff"] >= -0.005).all()
print(f"average of 5 n: blend3 {s.blend3.mean():.4f} | self-training {s.self_training.mean():.4f}")
print(f"rule: n=25 better in {wins25}/{TRIALS} seeds (need >= 8) | other n not worse by > 0.005: {others_ok}")
print("VERDICT:", "PASSES step 1 -> next: confirm in the dress rehearsal before adopting"
      if wins25 >= 8 and others_ok else "DOES NOT PASS -> keep frozen method; report as 'tried, did not help'")

,n_per_class,blend3,blend3_std,self_training,self_std,mean_diff,wins,pseudo_acc
0,5,0.5801,0.0401,0.5675,0.0351,-0.0126,1,0.8243
1,25,0.6412,0.0117,0.6372,0.0103,-0.0040,1,0.9075
2,50,0.6653,0.0121,0.6633,0.0096,-0.0020,3,0.9292
3,100,0.6827,0.0054,0.6831,0.0060,0.0004,6,0.9513
4,200,0.7028,0.0050,0.7011,0.0057,-0.0017,2,0.9561


average of 5 n: blend3 0.6544 | self-training 0.6504
rule: n=25 better in 1/10 seeds (need >= 8) | other n not worse by > 0.005: False
VERDICT: DOES NOT PASS -> keep frozen method; report as 'tried, did not help'


In [ ]:
# Saves a frozen copy of the working submission. Nothing is moved or deleted, only copied.
import shutil
from pathlib import Path

dst = Path("submission_frozen_v1")
assert not dst.exists(), "already saved, do not overwrite the safe copy"
dst.mkdir()
for folder in ["models", "results"]:                       # trained models + all result tables/plots
    shutil.copytree(folder, dst / folder)
(dst / "rehearsal").mkdir()
for f in Path("rehearsal").glob("dress_rehearsal*"):        # rehearsal results (not its 50 test models)
    shutil.copy2(f, dst / "rehearsal" / f.name)
for pattern in ["*.py", "*.ipynb", "*.md", "*.json", "features_*_v3.parquet"]:
    for f in Path(".").glob(pattern):                      # code, notebooks, docs, feature files
        shutil.copy2(f, dst / f.name)
print("saved:", sorted(p.name for p in dst.iterdir()))